In [88]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_groq import ChatGroq
import chromadb
import os

In [89]:
info_24 = r"C:\Users\sanke\Downloads\infosys-ar-24.pdf"
info_25 = r"C:\Users\sanke\Downloads\infosys-ar-25.pdf"
info_26 = r"C:\Users\sanke\Downloads\infosys-ar-26.pdf"

In [90]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [91]:
def ingest_pdf(pdf_path, report_year):

    reader = PdfReader(pdf_path)

    documents = []

    for page_no, page in enumerate(reader.pages, start=1):

        text = page.extract_text()

        if not text:
            continue

        text = text.strip()
        text = " ".join(text.split())

        chunks = splitter.split_text(text)

        for chunk in chunks:

            documents.append(
                Document(
                    page_content=chunk,
                    metadata={
                        "page_no": page_no,
                        "source": pdf_path,
                        "report": report_year
                    }
                )
            )

    return documents

In [92]:
docs_24 = ingest_pdf(info_24, "2023-24")
docs_25 = ingest_pdf(info_25, "2024-25")
docs_26 = ingest_pdf(info_26, "2025-26")

all_documents = docs_24 + docs_25 + docs_26

print("2023-24:", len(docs_24))
print("2024-25:", len(docs_25))
print("2025-26:", len(docs_26))
print("Total:", len(all_documents))

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/JLYLOK+MyriadPro-SemiCn', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(2364, 0, 2908809115984), '/LastChar': 122, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [196, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 559, 504, 518, 599, 450, 444, 583, 596, 227, 338, 501, 434, 742, 600, 0, 492, 616, 500, 453, 454, 590, 0, 0, 0, 0, 495, 0, 0, 0, 0, 0, 0, 447, 518, 404, 514, 460, 273, 511, 510, 219, 0, 434, 222, 770, 510, 499, 518, 0, 303, 364, 308, 505, 441, 685, 426, 434, 393]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/JLYLOK+MyriadPro-Semibold', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(2374, 0, 2908809

2023-24: 1484
2024-25: 1573
2025-26: 1685
Total: 4742


In [93]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4531.77it/s]


In [94]:
texts = [
    doc.page_content
    for doc in all_documents
]

vectors = embedding_model.encode(
    texts,
    show_progress_bar=True
)

print(vectors.shape)

Batches: 100%|██████████| 149/149 [02:22<00:00,  1.04it/s]

(4742, 384)


In [95]:
client = chromadb.Client()

collection = client.get_or_create_collection(
    name="infosys_financial_rag"
)

print("Collection created.")

Collection created.


In [96]:
ids = [
    f"chunk_{i}"
    for i in range(len(all_documents))
]

collection.add(
    ids=ids,
    documents=[
        doc.page_content
        for doc in all_documents
    ],
    embeddings=vectors.tolist(),
    metadatas=[
        doc.metadata
        for doc in all_documents
    ]
)

print("Documents stored:", collection.count())

Documents stored: 4742


In [97]:
question = "What was Infosys revenue in FY2025?"

query_vector = embedding_model.encode(
    question
).tolist()

result = collection.query(
    query_embeddings=[query_vector],
    n_results=5
)

for i in range(5):

    print("\nRank:", i + 1)
    print("Page:", result["metadatas"][0][i]["page_no"])
    print("Report:", result["metadatas"][0][i]["report"])
    print("Text:", result["documents"][0][i][:300])


Rank: 1
Page: 30
Report: 2024-25
Text: Infosys Integrated Annual Report 2024-25 58 Dear Members, The Board of Directors hereby submits the report of the business and operations of your Company (“the Company” or “Infosys”), along with the audited financial statements, for the financial year ended March 31, 2025. The consolidated performan

Rank: 2
Page: 29
Report: 2023-24
Text: Infosys Integrated Annual Report 2023-24 56 Dear Members, The Board of Directors hereby submits the report of the business and operations of your Company (“the Company” or “Infosys”), along with the audited financial statements, for the financial year ended March 31, 2024. The consolidated performan

Rank: 3
Page: 356
Report: 2025-26
Text: Infosys Integrated Annual Report 2025-26 387 Business segments Year ended March 31, 2026 and March 31, 2025: (In ` crore) Particulars Financial Services(1) Manufacturing Energy, Utilities, Resources and Services Retail(2) Communication(3) Hi-Tech Life Sciences(4) All other seg

In [120]:
def retrieve_documents(question):

    query_vector = embedding_model.encode(
        question
    ).tolist()

    result = collection.query(
        query_embeddings=[query_vector],
        n_results=5
    )

    retrieved = []

    for i in range(len(result["documents"][0])):

        retrieved.append({
            "text": result["documents"][0][i],
            "page_no": result["metadatas"][0][i]["page_no"],
            "source": result["metadatas"][0][i]["source"],
            "report": result["metadatas"][0][i]["report"],
            "distance": result["distances"][0][i]
        })

    return retrieved

In [121]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3597.06it/s]


In [122]:
def rerank_documents(question, documents):

    pairs = [
        (question, item["text"])
        for item in documents
    ]

    scores = reranker.predict(pairs)

    reranked = []

    for item, score in zip(documents, scores):

        reranked.append({
            "text": item["text"],
            "page_no": item["page_no"],
            "source": item["source"],
            "report": item["report"],
            "distance": item["distance"],
            "rerank_score": float(score)
        })

    reranked = sorted(
        reranked,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return reranked

In [123]:
def build_context(results):

    context = "\n\n---\n\n".join(
        f"Report: {item['report']}\n"
        f"Page: {item['page_no']}\n"
        f"Content: {item['text']}"
        for item in results[:3]
    )

    return context

In [124]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [125]:
def rag_answer(question):

    retrieved = retrieve_documents(question)

    reranked = rerank_documents(
        question,
        retrieved
    )

    context = build_context(
        reranked
    )

    prompt = f"""
You are a financial research assistant.

Answer the user's question using ONLY the provided context.

IMPORTANT:
- Pay close attention to the financial year requested in the question.
- Financial tables may contain values for multiple years.
- Match the answer to the EXACT year requested by the user.
- Do not use a value from another year.
- If multiple years are present, identify the correct year before answering.
- If the requested year's value cannot be determined from the context, say:
"I don't know based on the provided documents."

Do not use outside knowledge.
Do not make up information.
Do not mention sources, page numbers, citations, or references in your answer.

Question:
{question}

Context:
{context}
"""

    response = llm.invoke(prompt)

    sources = reranked[:3]

    return response.content, sources

In [126]:
def format_sources(sources):

    formatted = ["Sources:"]

    for source in sources:

        report = source["report"]

        formatted.append(
            f"- Infosys Annual Report {report} | "
            f"Page {source['page_no']}"
        )

    return "\n".join(formatted)

In [127]:
def final_response(question):

    answer, sources = rag_answer(question)

    source_text = format_sources(
        sources
    )

    return f"{answer}\n\n{source_text}"

In [128]:
question = "What was Infosys revenue in FY2025?"

print(
    final_response(question)
)

Infosys’ revenue for the financial year 2025 (ended 31 March 2025) was **₹1,62,990 crore** on a consolidated basis.

Sources:
- Infosys Annual Report 2024-25 | Page 30
- Infosys Annual Report 2023-24 | Page 29
- Infosys Annual Report 2025-26 | Page 33


In [129]:
question = "What was Infosys revenue in FY2026?"

result = retrieve_documents(question)

for i, item in enumerate(result, start=1):
    print(f"\nRank {i}")
    print("Report:", item["report"])
    print("Page:", item["page_no"])
    print("Distance:", item["distance"])
    print("Text:", item["text"][:500])


Rank 1
Report: 2024-25
Page: 30
Distance: 0.5792215466499329
Text: Infosys Integrated Annual Report 2024-25 58 Dear Members, The Board of Directors hereby submits the report of the business and operations of your Company (“the Company” or “Infosys”), along with the audited financial statements, for the financial year ended March 31, 2025. The consolidated performance of the Company and its subsidiaries has been referred to wherever required. 1. Results of our operations and state of affairs (in ` crore, except per equity share data) Particulars Standalone Conso

Rank 2
Report: 2023-24
Page: 29
Distance: 0.5833402872085571
Text: Infosys Integrated Annual Report 2023-24 56 Dear Members, The Board of Directors hereby submits the report of the business and operations of your Company (“the Company” or “Infosys”), along with the audited financial statements, for the financial year ended March 31, 2024. The consolidated performance of the Company and its subsidiaries has been referred to whe

In [130]:
question = "What was Infosys revenue in FY2026?"

result = retrieve_documents(question)
reranked = rerank_documents(question, result)

for i, item in enumerate(reranked, start=1):
    print(f"\nRank {i}")
    print("Report:", item["report"])
    print("Page:", item["page_no"])
    print("Rerank score:", item["rerank_score"])
    print("Text:", item["text"][:300])


Rank 1
Report: 2025-26
Page: 356
Rerank score: 2.60935115814209
Text: Infosys Integrated Annual Report 2025-26 387 Business segments Year ended March 31, 2026 and March 31, 2025: (In ` crore) Particulars Financial Services(1) Manufacturing Energy, Utilities, Resources and Services Retail(2) Communication(3) Hi-Tech Life Sciences(4) All other segments(5) Total Revenue 

Rank 2
Report: 2025-26
Page: 33
Rerank score: 2.3947291374206543
Text: Infosys Integrated Annual Report 2025-26 64 Dear Members, The Board of Directors hereby submits the report of the business and operations of your Company (“the Company” or “Infosys”), along with the audited financial statements, for the financial year ended March 31, 2026. The consolidated performan

Rank 3
Report: 2023-24
Page: 29
Rerank score: 2.374244451522827
Text: Infosys Integrated Annual Report 2023-24 56 Dear Members, The Board of Directors hereby submits the report of the business and operations of your Company (“the Company” or “Infosys”), 

In [131]:
question = "What was Infosys revenue in FY2026?"

print(final_response(question))

Infosys revenue from operations in FY 2026 was **₹1,78,650 crore**.

Sources:
- Infosys Annual Report 2025-26 | Page 356
- Infosys Annual Report 2025-26 | Page 33
- Infosys Annual Report 2023-24 | Page 29


In [132]:
#evaluation data set
evaluation_cases = [
    {
        "question": "What was the consolidated revenue of Infosys in FY2024?",
        "expected_answer": "₹1,53,670 crore",
        "report": "2023-24",
        "expected_page": 29
    },
    {
        "question": "What was the consolidated total income of Infosys in FY2024?",
        "expected_answer": "₹1,58,381 crore",
        "report": "2023-24",
        "expected_page": 29
    },
    {
        "question": "What was the consolidated revenue of Infosys in FY2025?",
        "expected_answer": "₹1,62,990 crore",
        "report": "2024-25",
        "expected_page": 30
    },
    {
        "question": "What was the consolidated revenue of Infosys in FY2026?",
        "expected_answer": "₹1,78,650 crore",
        "report": "2025-26",
        "expected_page": 33
    },
    {
        "question": "What was the consolidated employee benefit expense of Infosys in FY2024?",
        "expected_answer": "₹82,620 crore",
        "report": "2023-24",
        "expected_page": 324
    }
]

print("Evaluation cases:", len(evaluation_cases))

Evaluation cases: 5


In [133]:
def normalize_text(text):
    return (
        text.replace("\u202f", " ")
            .replace("\xa0", " ")
            .replace("₹ ", "₹")
            .lower()
            .strip()
    )

In [134]:
answer, sources = rag_answer(
    "What was the consolidated employee benefit expense of Infosys in FY2024?"
)

print(answer)

The consolidated employee benefit expense of Infosys in FY 2024 was **₹ 82,620 crore**.


In [135]:
correct = 0

for case in evaluation_cases:
    answer, sources = rag_answer(case["question"])

    generated = normalize_text(answer)
    expected = normalize_text(case["expected_answer"])

    is_correct = expected in generated

    if is_correct:
        correct += 1

    print("Question:", case["question"])
    print("Expected:", case["expected_answer"])
    print("Generated:", answer)
    print("Correct:", is_correct)
    print("-" * 80)

accuracy = correct / len(evaluation_cases)

print(f"Answer Accuracy: {correct}/{len(evaluation_cases)}")
print(f"Accuracy Score: {accuracy:.2%}")

Question: What was the consolidated revenue of Infosys in FY2024?
Expected: ₹1,53,670 crore
Generated: The consolidated revenue of Infosys in FY 2024 was **₹ 1,53,670 crore**.
Correct: True
--------------------------------------------------------------------------------
Question: What was the consolidated total income of Infosys in FY2024?
Expected: ₹1,58,381 crore
Generated: The consolidated total income of Infosys in FY 2024 was **₹ 1,58,381 crore**.
Correct: True
--------------------------------------------------------------------------------
Question: What was the consolidated revenue of Infosys in FY2025?
Expected: ₹1,62,990 crore
Generated: The consolidated revenue from operations for Infosys in FY 2025 was **₹ 1,62,990 crore**.
Correct: True
--------------------------------------------------------------------------------
Question: What was the consolidated revenue of Infosys in FY2026?
Expected: ₹1,78,650 crore
Generated: The consolidated revenue for Infosys in FY 2026 was **₹1,

In [136]:
def calculate_recall_at_k(retrieved_pages, expected_page, k=3):
    return int(expected_page in retrieved_pages[:k])

In [137]:
recall_scores = []

for case in evaluation_cases:
    result = retrieve_documents(case["question"])

    retrieved_pages = [item["page_no"] for item in result]

    score = calculate_recall_at_k(
        retrieved_pages,
        case["expected_page"],
        k=3
    )

    recall_scores.append(score)

    print("Question:", case["question"])
    print("Expected page:", case["expected_page"])
    print("Retrieved pages:", retrieved_pages[:3])
    print("Recall@3:", score)
    print("-" * 80)

recall_at_3 = sum(recall_scores) / len(recall_scores)

print(f"Recall@3: {recall_at_3:.2%}")

Question: What was the consolidated revenue of Infosys in FY2024?
Expected page: 29
Retrieved pages: [30, 29, 33]
Recall@3: 1
--------------------------------------------------------------------------------
Question: What was the consolidated total income of Infosys in FY2024?
Expected page: 29
Retrieved pages: [30, 29, 33]
Recall@3: 1
--------------------------------------------------------------------------------
Question: What was the consolidated revenue of Infosys in FY2025?
Expected page: 30
Retrieved pages: [30, 29, 33]
Recall@3: 1
--------------------------------------------------------------------------------
Question: What was the consolidated revenue of Infosys in FY2026?
Expected page: 33
Retrieved pages: [30, 29, 33]
Recall@3: 1
--------------------------------------------------------------------------------
Question: What was the consolidated employee benefit expense of Infosys in FY2024?
Expected page: 324
Retrieved pages: [287, 256, 324]
Recall@3: 1
--------------------

In [138]:
def calculate_mrr_at_k(retrieved_pages, expected_page, k=3):
    for rank, page in enumerate(retrieved_pages[:k], start=1):
        if page == expected_page:
            return 1 / rank

    return 0

In [139]:
mrr_scores = []

for case in evaluation_cases:
    result = retrieve_documents(case["question"])

    retrieved_pages = [item["page_no"] for item in result]

    score = calculate_mrr_at_k(
        retrieved_pages,
        case["expected_page"],
        k=3
    )

    mrr_scores.append(score)

    print("Question:", case["question"])
    print("Expected page:", case["expected_page"])
    print("Retrieved pages:", retrieved_pages[:3])
    print("MRR@3:", score)
    print("-" * 80)

mrr_at_3 = sum(mrr_scores) / len(mrr_scores)

print(f"MRR@3: {mrr_at_3:.2%}")

Question: What was the consolidated revenue of Infosys in FY2024?
Expected page: 29
Retrieved pages: [30, 29, 33]
MRR@3: 0.5
--------------------------------------------------------------------------------
Question: What was the consolidated total income of Infosys in FY2024?
Expected page: 29
Retrieved pages: [30, 29, 33]
MRR@3: 0.5
--------------------------------------------------------------------------------
Question: What was the consolidated revenue of Infosys in FY2025?
Expected page: 30
Retrieved pages: [30, 29, 33]
MRR@3: 1.0
--------------------------------------------------------------------------------
Question: What was the consolidated revenue of Infosys in FY2026?
Expected page: 33
Retrieved pages: [30, 29, 33]
MRR@3: 0.3333333333333333
--------------------------------------------------------------------------------
Question: What was the consolidated employee benefit expense of Infosys in FY2024?
Expected page: 324
Retrieved pages: [287, 256, 324]
MRR@3: 0.333333333333

In [140]:
question = "What was the consolidated total income of Infosys in FY2024?"

result = retrieve_documents(question)

for i, item in enumerate(result, start=1):
    print(f"Rank {i}")
    print(f"Report: {item['report']}")
    print(f"Page: {item['page_no']}")
    print(f"Distance: {item['distance']}")
    print(f"Text: {item['text'][:400]}")
    print("-" * 80)

Rank 1
Report: 2024-25
Page: 30
Distance: 0.4705982208251953
Text: Infosys Integrated Annual Report 2024-25 58 Dear Members, The Board of Directors hereby submits the report of the business and operations of your Company (“the Company” or “Infosys”), along with the audited financial statements, for the financial year ended March 31, 2025. The consolidated performance of the Company and its subsidiaries has been referred to wherever required. 1. Results of our ope
--------------------------------------------------------------------------------
Rank 2
Report: 2023-24
Page: 29
Distance: 0.4897209703922272
Text: Infosys Integrated Annual Report 2023-24 56 Dear Members, The Board of Directors hereby submits the report of the business and operations of your Company (“the Company” or “Infosys”), along with the audited financial statements, for the financial year ended March 31, 2024. The consolidated performance of the Company and its subsidiaries has been referred to wherever required. 1. Re

In [141]:
print(result[1]["text"])

Infosys Integrated Annual Report 2023-24 56 Dear Members, The Board of Directors hereby submits the report of the business and operations of your Company (“the Company” or “Infosys”), along with the audited financial statements, for the financial year ended March 31, 2024. The consolidated performance of the Company and its subsidiaries has been referred to wherever required. 1. Results of our operations and state of affairs (In ` crore, except per equity share data) Particulars Standalone Consolidated For the year ended March 31, YoY growth (%) For the year ended March 31, YoY growth (%)2024 2023 2024 2023 Revenue from operations 1,28,933 1,24,014 4.0 1,53,670 1,46,767 4.7 Other income, net* 7, 417 3,859 92.2 4,711 2,701 74.4 Total income 1,36,350 1, 27, 873 6.6 1,58,381 1,49,468 6.0 Expenses Cost of sales 89,032 85,762 3.8 1, 07, 413 1,02,353 4.9 Selling and marketing expenses 5,668 5,018 13.0 6,973 6,249 11. 6 General and administration expenses 5,420 5,293 2.4 7, 537 7, 26 0 3.8
